# 实验二：图像增强与滤波

由已有 exp2.py 分步整理。原脚本与四张历史截图均保留；噪声参数和滤波算法保持原样。按顺序运行所有单元。随机噪声未固定种子，每次结果有差异；截图是原实验成果，并非本 notebook 本次执行输出。

## 0. 环境与路径
依赖：opencv-python、numpy、scikit-image、matplotlib。缺少依赖时在当前 notebook 内核环境安装：
```python
%pip install -r requirements.txt
```
手动中值滤波包含逐像素循环，运行时间比库函数长。

In [ ]:
from pathlib import Path
from datetime import datetime
ROOT = Path.cwd()
if not (ROOT / 'p1.jpg').exists() and (ROOT / '2nd.exp/p1.jpg').exists():
    ROOT = ROOT / '2nd.exp'
INPUT = ROOT / 'p1.jpg'
if not INPUT.exists():
    raise FileNotFoundError('请从 2nd.exp 文件夹打开 notebook，或将工作目录设置为仓库根目录')
GENERATED = ROOT / 'generated' / datetime.now().strftime('%Y%m%d-%H%M%S-%f')
GENERATED.mkdir(parents=True, exist_ok=False)
print('本次结果目录:', GENERATED)


## 1. 读取图像、查看像素与灰度转换

In [ ]:
import cv2
import numpy as np
from skimage.util import random_noise
from matplotlib import pyplot as plt

# 1. 读取原始图像
img = cv2.imdecode(np.fromfile(INPUT, dtype=np.uint8), cv2.IMREAD_COLOR)

# 检查图像是否读取成功
if img is None:
    raise FileNotFoundError("没有找到 p1.jpg，请检查图片路径和文件名")

# 2. 查看图像尺寸
print("图像尺寸（高、宽、通道）:", img.shape)

# 3. 获取 [100, 100] 位置的像素值
b, g, r = img[100, 100]
print("该像素的 BGR 值:", b, g, r)
print("该像素的 RGB 值:", r, g, b)

# 4. 将 BGR 转换为 RGB
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 5. 转换为灰度图
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 6. 显示原图和灰度图
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gray_img, cmap="gray")
plt.title("Gray Image")
plt.axis("off")

plt.tight_layout()
plt.savefig(GENERATED / "gray_comparison.png", dpi=300, bbox_inches="tight")
plt.show()



## 2. 添加椒盐噪声和高斯噪声

In [ ]:
# ==================== 添加噪声 ====================

# 添加椒盐噪声
sp_noise_img = random_noise(
    rgb_img,
    mode='s&p',
    amount=0.4
)
# 添加高斯噪声
gus_noise_img = random_noise(
    rgb_img,
    mode='gaussian',
    mean=0.2,
    var=0.03
)
# ==================== 显示噪声对比图 ====================
plt.figure(figsize=(12, 4))
# 原图
plt.subplot(1, 3, 1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis("off")
# 椒盐噪声图
plt.subplot(1, 3, 2)
plt.imshow(sp_noise_img)
plt.title("Salt & Pepper Noise")
plt.axis("off")
# 高斯噪声图
plt.subplot(1, 3, 3)
plt.imshow(gus_noise_img)
plt.title("Gaussian Noise")
plt.axis("off")

plt.tight_layout()
plt.savefig(GENERATED / "noise_comparison.png", dpi=300, bbox_inches="tight")
plt.show()




## 3. 均值、中值和高斯滤波对比

In [ ]:
# ==================== 图像滤波 ====================

# 将噪声图像转换为 0~255 的 uint8 格式
sp_noise_u8 = (sp_noise_img * 255).clip(0, 255).astype(np.uint8)
gus_noise_u8 = (gus_noise_img * 255).clip(0, 255).astype(np.uint8)

# 1. 均值滤波
mean_sp = cv2.blur(sp_noise_u8, (5, 5))
mean_gus = cv2.blur(gus_noise_u8, (5, 5))

# 2. 中值滤波
mid_sp = cv2.medianBlur(sp_noise_u8, 5)
mid_gus = cv2.medianBlur(gus_noise_u8, 5)

# 3. 高斯滤波
gauss_sp = cv2.GaussianBlur(sp_noise_u8, (5, 5), 0)
gauss_gus = cv2.GaussianBlur(gus_noise_u8, (5, 5), 0)

# ==================== 显示滤波对比图 ====================

plt.figure(figsize=(14, 8))

# 第一行：椒盐噪声
plt.subplot(2, 3, 1)
plt.imshow(mean_sp)
plt.title("S&P Noise - Mean Filter")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(mid_sp)
plt.title("S&P Noise - Median Filter")
plt.axis("off")

plt.subplot(2, 3, 3)
plt.imshow(gauss_sp)
plt.title("S&P Noise - Gaussian Filter")
plt.axis("off")

# 第二行：高斯噪声
plt.subplot(2, 3, 4)
plt.imshow(mean_gus)
plt.title("Gaussian Noise - Mean Filter")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(mid_gus)
plt.title("Gaussian Noise - Median Filter")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(gauss_gus)
plt.title("Gaussian Noise - Gaussian Filter")
plt.axis("off")

plt.tight_layout()
plt.savefig(GENERATED / "filter_comparison.png", dpi=300, bbox_inches="tight")
plt.show()





## 4. 手动实现中值滤波

In [ ]:
# ==================== 手动实现中值滤波 ====================

def manual_median_filter_color(image, kernel_size=5):
    # 计算窗口向外扩展的像素数量
    pad = kernel_size // 2

    # 创建与原图形状相同的空图像
    filtered_img = np.zeros_like(image)

    # 对 RGB 三个通道分别处理（输入为 RGB 图像）
    for c in range(3):
        channel = image[:, :, c]

        # 对图像边缘进行填充
        padded_channel = np.pad(
            channel,
            pad_width=pad,
            mode="edge"
        )

        # 遍历图像中的每一个像素
        for i in range(channel.shape[0]):
            for j in range(channel.shape[1]):

                # 提取当前像素周围的 5×5 窗口
                region = padded_channel[
                    i:i + kernel_size,
                    j:j + kernel_size
                ]

                # 计算窗口内像素的中位数
                filtered_img[i, j, c] = np.median(region)

    return filtered_img


# 对椒盐噪声图像进行手动中值滤波
manual_mid = manual_median_filter_color(
    sp_noise_u8,
    kernel_size=5
)

# ==================== 显示处理前后的对比 ====================

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(sp_noise_u8)
plt.title("Salt & Pepper Noise")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(manual_mid)
plt.title("Manual Median Filter")
plt.axis("off")

plt.tight_layout()
plt.savefig(
    GENERATED / "manual_median_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 5. 保存独立图像与检查手动实现
OpenCV 中值滤波使用边缘复制，与手动函数的 edge 填充一致。这里比较相同输入和相同窗口下的结果。

In [ ]:
def save_rgb(name, array):
    bgr = cv2.cvtColor(array, cv2.COLOR_RGB2BGR)
    ok, encoded = cv2.imencode('.png', bgr)
    if not ok:
        raise RuntimeError('图片编码失败')
    encoded.tofile(GENERATED / name)

save_rgb('rgb_image.png', rgb_img)
save_rgb('original_bgr.png', rgb_img)  # 文件存储使用 BGR 编码，查看时显示正常颜色
ok, encoded = cv2.imencode('.png', gray_img)
if not ok:
    raise RuntimeError('灰度图编码失败')
encoded.tofile(GENERATED / 'gray_image.png')
save_rgb('manual_median.png', manual_mid)
diff = np.abs(manual_mid.astype(np.int16) - mid_sp.astype(np.int16))
print('手动与 OpenCV 中值滤波最大绝对差:', diff.max())
assert np.array_equal(manual_mid, mid_sp), '结果不一致，请检查窗口和边界处理'
